# Proyecto EDPN: ¿Podemos escuchar un grafo?

### Diego Echeverria, Luciano Avegno, Santiago Escobar

Los resultados vistos estan basados en el paper:

"Hearing the clusters of a graph: A distributed algorithm"

por Tuhin Sahai, Alberto Speranzon, Andrzej Banaszuk

## Librerias


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## datos de grafos
import networkx as nx
from scipy.fft import fft, rfft
from scipy.signal import find_peaks

## animaciones
import matplotlib.animation as animation
from IPython.display import HTML


## Funciones previas

In [ ]:
# Este es para dibujar un grafo
def dibujar_grafo(A):

  plt.clf()

  G = nx.Graph(A)
  layout = nx.spring_layout(G)
  labels = nx.get_edge_attributes(G, "weight")
  nx.draw(G, pos = layout)
  nx.draw_networkx_edge_labels(G, pos = layout, edge_labels = labels)

  plt.show()

In [ ]:
# Este es una función que calcula el laplaciano
# Basado en la pagina 2 como  L = I - D^-1 W
def laplacian(A):
  n,m = A.shape
  D = np.diag(sum(A))
  L = np.eye(n) - np.dot(np.linalg.inv(D),A)
  return L

In [ ]:
# Iterador del algoritmo 3.1  (el while)
def iteracion(u,c,L):

  # tamaño de u
  n,m = np.shape(u)

  # matraca
  x = -u[:, -2] + np.dot( (2*np.eye(n) - c*L),  u[:,-1])
  u = np.append(u, x.reshape(n,1), axis=1)
  return u


## Inicialización de lo valores para ejemplo de un ciclo

In [ ]:
n = 10            # cantidad de nodos
peso_corte = 0.1  # el peso bajo
peso_otro = 1.0   # el peso alto

A = np.zeros( (n,n) ) # matriz inicial

for i in range(n-1):
  A[i, i+1] = peso_otro
A[n-1,0] = peso_otro

# añadir los cortes ( sobreescribir )

A[n//2 - 1,  n//2] = peso_corte
A[n-1, 0] = peso_corte

# simetrizar la matriz
A = A + A.T

In [ ]:
print("Matriz de ciclos")
print(A)
dibujar_grafo(A)

Se partió con un ejemplo de un grafo cíclico de 8 vértices, en donde 2 aristas tienen un ''peso'' mucho menor al resto, dividiendo claramente a los vértices en dso grupos distinguibles por su conectividad, para así verificar que efectivamente al ejecutar el algoritmo este divide al grafo en dos clusters. Este ejemplo en particular está inspirado en el paper original, y fue discutido con el profesor Axel Osses en una reunión.

In [ ]:
## inicio de hiperparametros
n,m = A.shape
c = 1.5  # valor de c que debe estar  entre 0 y 2  ( paper dice raiz de 2, pero este c ya esta al cuadrado)
T_max = n**2 + 10000 # Este valor se asume superior al tiempo de mezcla, la suma de 10mil nos asegura convergencia en grafos pequeños
k = 2 # cantidad de valores que se quieren ver

In [ ]:

def FuncionOndas(c,T_max,A,k=2):

  n,m = A.shape  # tamaño del grafo
  u = np.random.random( (n,1)) # valores iniciales random
  u = np.append(u,u, axis=1)

  # Saco laplaciano
  L = laplacian(A)

  for i in range(T_max):
    u = iteracion(u,c,L)

  # hago una lista de clusternumber
  ClusterNumber = []

  # Por cada nodo
  for i in range(n):

    fourier = fft(u[i,2:]) #fourier
    amplitudes = np.abs(fourier[1 : T_max//2]) #amplitudes
    peaks, _ = find_peaks(amplitudes) #peaks
    peaks = np.sort(peaks) # sort de peaks

    cluster_id = 0 # futuro numero de cluster

    for j in range(k - 1): # busqueda de k frecuencias

        if j < len(peaks):

            peak_idx = peaks[j] + 1 # 1 porque cortamos el índice 0 al buscar picos
            coef = fourier[peak_idx].real # coeficiente

            if coef > 0: # mirar signo
                signature = 1
            else:
                signature = 0
        else:
            signature = 0
        cluster_id += signature << j # esto es del paper como la suma por 2^{j-1}

    ClusterNumber.append(cluster_id) # anotar valor

  print(ClusterNumber)
  return ClusterNumber



In [ ]:
FuncionOndas(c,T_max,A,k=2)

## Ejemplo con grafo completo

In [ ]:
# Crear matriz de adjacencia de un grafo completo
n = 10
peso_corte = 1
peso = 10

A = np.zeros((n,n))

# Relleno con unos en las aristas normales
for i in range(n//2):
    for j in range(i+1,n//2):
        A[i,j] = A[j,i] = peso

        A[i+n//2,j+n//2] = A[j+n//2,i+n//2] = peso
for i in range(n//2):
    for j in range(i,n//2):
        A[i,j+n//2] = A[j,i+n//2] = peso_corte
        A[i+n//2,j] = A[j+n//2,i] = peso_corte


print("Matriz del bipartito:")
# Puede ser que sea necesario ejecutar varias veces este bloque para que el gráfico se muestre bien
dibujar_grafo(A)

In [ ]:
FuncionOndas(c,T_max,A,k=2)

Este es otro ejemplo de grafo, ahora se trabaja con un grafo completo, en donde hay dos clusters unidos por aristas con un peso mucho menor que las que conectan a los vértices entre si.

### Ahora voy a usar el networkx para pasar grafos conocidos a matriz de adyacencia

In [ ]:
# Grafica un grafo pintando los nodos del color de los clusters
def graficador(G, pos, clusters):

  plt.clf() # clear del graficador

  # diferencias aristas fuertes y debiles para pintar distinto
  inter_edges = [(u, v) for u, v in G.edges() if clusters[list(G.nodes()).index(u)] != clusters[list(G.nodes()).index(v)]]
  intra_edges = [(u, v) for u, v in G.edges() if clusters[list(G.nodes()).index(u)] == clusters[list(G.nodes()).index(v)]]
  nx.draw_networkx_edges(G, pos, edgelist=intra_edges, alpha=0.5, edge_color= "green")
  nx.draw_networkx_edges(G, pos, edgelist=inter_edges, style="dashed", edge_color="blue", alpha=0.5)

  # Dibujar nodos diferenciado por clusters
  nx.draw_networkx_nodes(G, pos, node_color=clusters, cmap=plt.cm.Set1, node_size=300)
  nx.draw_networkx_labels(G, pos, font_size=10)

  plt.title("FFT clustering")
  plt.axis("off")
  plt.show()


## Tambien una función para ver animaciones de los resultados

In [ ]:

### Animador

def Animador(G, ClusterNumber, layout):

  pos = layout
  fig, ax = plt.subplots(figsize=(7, 7))

  ClusterNumber = [ x+1 for x in ClusterNumber] # Hago tapa para los clusternumber 0

  # Fija los límites del gráfico para que la cámara no salte al vibrar las cuerdas
  x_vals = [p[0] for p in pos.values()]
  y_vals = [p[1] for p in pos.values()]
  ax.set_xlim(min(x_vals) - 0.2, max(x_vals) + 0.2)
  ax.set_ylim(min(y_vals) - 0.2, max(y_vals) + 0.2)
  ax.axis("off")

  # Colorear los nodos basándonos en a qué clúster pertenecen para que se vea mejor
  nx.draw_networkx_nodes(G, pos, ax=ax, node_color=ClusterNumber, cmap=plt.cm.Set3, node_size=300)
  nx.draw_networkx_labels(G, pos, ax=ax)

  # Creamos los objetos de las líneas (cuerdas) vacíos
  lineas_aristas = []
  edges_list = list(G.edges())

  for _ in edges_list:
      line, = ax.plot([], [], color='gray', lw=2)
      lineas_aristas.append(line)


  def actualizar_animation(frame, clustername, lineas_aristas, edges_list, pos):
      s = np.linspace(0, 1, 50)

      # ----------------------------------------------------
      # CONTROL DE VELOCIDAD (CÁMARA LENTA)
      # Baja este número si quieres que vaya aún más lento
      velocidad_camara_lenta = 0.025
      # ----------------------------------------------------

      for idx, (nodo1, nodo2) in enumerate(edges_list):
          x1, y1 = pos[nodo1]
          x2, y2 = pos[nodo2]

          dx = x2 - x1
          dy = y2 - y1
          distancia = np.hypot(dx, dy)

          if distancia == 0: continue

          nx_norm = -dy / distancia
          ny_norm = dx / distancia

          x_base = x1 + s * dx
          y_base = y1 + s * dy
          amplitud_maxima = 0.15 * distancia
          # Frecuencia promedio de la arista
          if clustername[nodo1] == clustername[nodo2]:
              freq = 1 
              if clustername[nodo1] == 1:
                  desplazamiento = amplitud_maxima * np.sin(np.pi * s) * np.sin(2 * np.pi * freq * frame * velocidad_camara_lenta)
              else:
                  desplazamiento = amplitud_maxima * np.sin(np.pi * s) * np.sin(2 * np.pi * freq * frame * velocidad_camara_lenta + np.pi)
          else:
              freq = 0
              desplazamiento = amplitud_maxima * np.sin(np.pi * s) * np.sin(2 * np.pi * freq * frame * velocidad_camara_lenta)
          #freq = (clustername[nodo1] + clustername[nodo2]) / 2.0

          # FÓRMULA CORREGIDA DE LA CUERDA VIBRANTE
          
          

          x_curve = x_base + desplazamiento * nx_norm
          y_curve = y_base + desplazamiento * ny_norm

          lineas_aristas[idx].set_data(x_curve, y_curve)

      ax.set_title(f"Vibración de Cuerdas (t={frame})")
      return lineas_aristas

  # --- 4. RENDERIZADO ---
  ani = animation.FuncAnimation(
      fig,
      actualizar_animation,
      frames=400, # Aumenté los frames a 400 para que la cámara lenta dure más tiempo en pantalla
      interval=30, # Corre a unos 33 FPS para más fluidez
      blit=False,
      fargs=(ClusterNumber, lineas_aristas, edges_list, pos)
  )

  plt.close()
  return ani


# Ahora vamos a ver ejemplos tipicos

### Grafo Ciclo

In [ ]:


n = 10            # cantidad de nodos
peso_corte = 0.1  # el peso bajo
peso_otro = 1.0   # el peso alto
c = 1.9
T_max = n**2 +1000

A = np.zeros( (n,n) ) # matriz inicial

for i in range(n-1):
  A[i, i+1] = peso_otro
A[n-1,0] = peso_otro

# añadir los cortes ( sobreescribir )

A[n//2 - 1,  n//2] = peso_corte
A[n-1, 0] = peso_corte

# simetrizar la matriz
A = A + A.T

#dibujar_grafo(A)
ClusterNumber = FuncionOndas(c,T_max,A,k=2)

G = nx.Graph(A)

graficador(G, nx.spring_layout(G) , ClusterNumber)


En este resultado se puede ver que efectivamente, se clasifican los clusters del grafo en dos grupos cuya conectividad es mayor entre sí (en el sentido que el peso de sus aristas es mayor entre estos que para el otro grupo)

In [ ]:
# retornar animacion
ani = Animador(G, ClusterNumber, nx.spring_layout(G))
HTML(ani.to_jshtml())

En ésta animación se aprecia que los vértices de un mismo clúster comparten un mismo modo de vibración, en cambio, en las aristas que cruzan los grupos de vértices se produce una suerte de interferencia entre estos modos de la onda.

### Grafo completo

In [ ]:

n = 20 # nodos
G = nx.complete_graph(n)
c = 1.90
T_max = n**(2) + 10000 # iteraciones
cluster1 = [2,3] # estos son los nodos que arbitrariamente se deciden como los del otro clustes
for u,v in G.edges():

    if u in cluster1 and v not in cluster1:
        G[u][v].update(weight=0.1)
    elif u not in cluster1 and v in cluster1:
        G[u][v].update(weight=0.1)
    else:
        G[u][v].update(weight=10)

A = nx.to_numpy_array(G)
ClusterNumber = FuncionOndas(c,T_max,A,k=2)
dibujar_grafo(A)
graficador(G, nx.spring_layout(G) , ClusterNumber)


  Acá pasamos a un caso más extremo: hay un grafo completo donde un clúster de vértices es mucho más grande que el otro, y son conectados por una gran cantidad de aristas de peso mucho menor (de 2 órdenes de magnitud más pequeño) que las que unen vértices de un mismo clúster. En este caso, usamos el mismo tiempo de mezcla que con el caso de ciclo, y comienzan a aparecer pequeños errores en la clasificación de clústers, lo que se explica en el resultado siguiente.

In [ ]:
n = 6 # nodos
G = nx.complete_graph(n)
c = 1.90
T_max = n**(2) + 10000 # iteraciones
cluster1 = [2,3] # estos son los nodos que arbitrariamente se deciden como los del otro clustes
for u,v in G.edges():

    if u in cluster1 and v not in cluster1:
        G[u][v].update(weight=1)
    elif u not in cluster1 and v in cluster1:
        G[u][v].update(weight=1)
    else:
        G[u][v].update(weight=5)

A = nx.to_numpy_array(G)
ClusterNumber = FuncionOndas(c,T_max,A,k=2)
dibujar_grafo(A)
graficador(G, nx.spring_layout(G) , ClusterNumber)

ani = Animador(G, ClusterNumber, nx.spring_layout(G))
HTML(ani.to_jshtml())

En esta grafo se ve que la estrategia de clasificación de clústers mediante el uso del laplaciano, no es estable cuando el orden de magnitud en la diferencia de las aristas es de dos órdenes. Esto tiene sentido, pues aparecen fenómenos de refracción que ocurren cuando la onda pasa por una arista de mayor peso a una con peso mucho menor, lo que evita que sean claros los modos de la onda en el valor del vector propio.

### Grafo de malla

In [ ]:
B = np.array([[0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0.1,0,0,0,0,0,0.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0.1,0,0,0,0,0,0.1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0.1,0,0,0,0,0,0.1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.1,0,0,0,0,0,0.1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0.1,0,0,0,0,0,0.1,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.1,0,0,0,0,0,0.1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0.1,0,0,0,0,1,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.1,0,0,0,0,0,0.1,0,1,0,0,0,0,1,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0],
[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0]])



# Simetrizar B
#B = B + B.T
print("Grafo Malla")
print(B)
Malla = nx.Graph(B)
grilla = {0: [1,6],
          1: [2,6],
          2: [3,6],
          3: [4,6],
          4: [5,6],
          5: [0,5],
          6: [1,5],
          7: [2,5],
          8: [3,5],
          9: [4,5],
          10: [5,5],
          11: [6,5],
          12: [0,4],
          13: [1,4],
          14: [2,4],
          15: [3,4],
          16: [4,4],
          17: [5,4],
          18: [6,4],
          19: [0,3],
          20: [1,3],
          21: [2,3],
          22: [3,3],
          23: [4,3],
          24: [5,3],
          25: [6,3],
          26: [0,2],
          27: [1,2],
          28: [2,2],
          29: [3,2],
          30: [4,2],
          31: [5,2],
          32: [6,2],
          33: [0,1],
          34: [1,1],
          35: [2,1],
          36: [3,1],
          37: [4,1],
          38: [5,1],
          39: [6,1],
          40: [1,0],
          41: [2,0],
          42: [3,0],
          43: [4,0],
          44: [5,0],
          }
rotulos = nx.get_edge_attributes(Malla, "weight")


In [ ]:
## inicio de hiperparametros para B
n,m = B.shape
c = 1.99  # valor de c que debe estar  entre 0 y 2  ( paper dice raiz de 2, pero este c ya esta al cuadrado)
T_max = n**2 + 10000 # Este valor se asume superior al tiempo de mezcla
k = 2 # cantidad de valores que se quieren ver
clusternames = FuncionOndas(c,T_max,B,k=2)
graficador(Malla, grilla, clusternames)

In [ ]:
ani = Animador(Malla,clusternames, grilla)
HTML(ani.to_jshtml())

En el grafo de tipo grilla, las conexiones entre cada lado del grafo son aristas débiles, por lo que se clasifican correctamente ambos clústers del grafo, incluso cuando estos grupos de vértices son más grandes que en el caso del grafo cíclico. En las animaciones se ve que en las aristas que apuntan solo a vértices de un mismo grupo oscilan con la misma frecuencia.

### Grafo estrella

En este podemos notar que por la estructura del grafo no fue posible que el algoritmo encontrara clusters correctamente

In [ ]:

n = 10
c = 1.90
T_max = n**2 +10000
Grafo = nx.star_graph(n)

vertices_debil = [1]
## ahora creo las verticess debiles
for u,v in Grafo.edges():

    if u in vertices_debil and v not in vertices_debil:
        Grafo[u][v].update(weight=1)
    elif u not in vertices_debil and v in vertices_debil:
        Grafo[u][v].update(weight=1)
    else:
        Grafo[u][v].update(weight=100)

#Grafo.add_edge(1,2, weight=100)

A = nx.to_numpy_array(Grafo)
clusternames = FuncionOndas(c,T_max,A,k=2)
dibujar_grafo(A)
graficador(Grafo, nx.spring_layout(Grafo), clusternames)

# Parece que un grafo donde los clusters son solo un solo nodo o donde es este formato estrella
# tenemos que el algoritmo no se comporta correctamente

Acá se aprecia que en el grafo tipo estrella, la clasificación falla. Esto deber ser consecuencia de que hay tres clústers con vértices que tienen pesos similares: 2 vértices con baja conectividad al del centro, y los que están unidos por aristas de mayor peso al centro. Realizar pruebas con el 3er vector propio podría solucionar es

### Grafo Bipartito

In [ ]:

n = 10
c = 1.90
T_max = n**2 +20000
Grafo = nx.bipartite.complete_bipartite_graph(n,n)
top, bottom = nx.bipartite.sets(Grafo)

arista_debil = top
## ahora creo las aristas debiles
for u,v in Grafo.edges():

    if u in arista_debil and v not in arista_debil:
        Grafo[u][v].update(weight=0.1)

# En este paso añado las aristas entre las partes bipartitas
# esto es para tener clusters conexos
for u in top:
  for v in top:
    if u != v:
      Grafo.add_edge(u,v,weight=10)

for u in bottom:
  for v in bottom:
    if u != v:
      Grafo.add_edge(u,v,weight=10)

A = nx.to_numpy_array(Grafo)
clusternames = FuncionOndas(c,T_max,A,k=2)

########### codigo para graficar con layout bipartito
layout = nx.bipartite_layout(Grafo,top)
labels = nx.get_edge_attributes(Grafo, "weight")
nx.draw(Grafo, pos = layout)
nx.draw_networkx_edge_labels(Grafo, pos = layout, edge_labels = labels)
nx.draw_networkx_nodes(Grafo, pos=layout, nodelist=top, node_color="r")
nx.draw_networkx_nodes(Grafo, pos=layout, nodelist=bottom, node_color="b")
plt.show()
###########



print("los elementos de nodos top es: ", top)
graficador(Grafo, nx.bipartite_layout(Grafo,top), clusternames)

# Parece que un grafo donde los clusters son solo un solo nodo o donde es este formato estrella
# tenemos que el algoritmo no se comporta correctamente

Acá se puede ver que en el caso de un grafo ''bipartito'' donde las conexiones, aunque numerosas entre ambos lados del grafo, son más débiles que entre vértices de un mismo ''lado''. Esto se ve reflejado claramente en los clusters generados por el algoritmo trabajado.

### Grafo con corte


In [ ]:
## Voy a crear dos grafos con erdos renyi y despues los voy a conectar
import random

# Tamaños de ambos grafos y probabilidad de aristas
n1,p1 = 300, 0.5
n2,p2 = 300, 0.4

# hiperparamentros
c = 1.90
T_max = (n1+n2) +10000

# grafos
Grafo1 = nx.erdos_renyi_graph(n1,p1)
Grafo2 = nx.erdos_renyi_graph(n2,p2)
Grafo3 = nx.disjoint_union(Grafo1, Grafo2)
# aristas fuertes
nx.set_edge_attributes(Grafo, 10, name="weight")

# sampleo aleatorio de los dos clusters
nodosG1 = random.sample(range(len(Grafo1)), 5)
nodosG2 = random.sample(range(len(Grafo1), len(Grafo1) + len(Grafo2)), 5)

# creacion de uniones debiles
new_edges = list(zip(nodosG1, nodosG2))
Grafo3.add_edges_from(new_edges, weight=0.1)

A = nx.to_numpy_array(Grafo3)
clusternames = FuncionOndas(c,T_max,A,k=2)
dibujar_grafo(A)
graficador(Grafo3, nx.spring_layout(Grafo3), clusternames)

En estos gráficos ya se comienzan a trabajar con grafos aleatorios del tipo Erdös–Rényi, con probabilidades $p_1 = 0.5$ y $p_2 = 0.4$. Luego, estos grafos son conectados entre sí por aristas con un peso menor a las aristas del grafo. Como se ve en los resultados, los clústers encontrados no son los esperados (se espera que los clústers 1 y 2 sean cada uno los vértices de los grafos aleatorios antes de ser unidos por aristas). Esto puede ser

### barabasi_albert

In [ ]:
n_comunidad = 40
m_aristas = 2

peso_interno = 10000
peso_corte = 100

var = 30

G1 = nx.barabasi_albert_graph(n_comunidad, m_aristas)
G2 = nx.barabasi_albert_graph(n_comunidad, m_aristas)

mapping = {i: i+n_comunidad for i in range(n_comunidad)}
G2 = nx.relabel_nodes(G2, mapping)

G_bipartito = nx.union(G1,G2)

for u,v in G_bipartito.edges():
  G_bipartito[u][v]['weight'] = random.randint(peso_interno-var,peso_interno+var)


# sampleo aleatorio de los dos clusters
nodosG1 = random.sample(range(len(G1)), 9)
nodosG2 = random.sample(range(len(G1), len(G1) + len(G2)), 9)

# creacion de uniones debiles
new_edges = list(zip(nodosG1, nodosG2))
G_bipartito.add_edges_from(new_edges, weight=  random.randint(peso_corte-var,peso_corte+var))
A_bipartito = nx.to_numpy_array(G_bipartito)

c = 1.95
T_max = n_comunidad**2 + 1000
k = 2

ClusterNumber = FuncionOndas(c,T_max, A_bipartito,k=2)
pos_bip = nx.spring_layout(G_bipartito, k=0.15, iterations=50, seed=42)
graficador(G_bipartito, pos_bip, ClusterNumber)

#ani = Animador(G_bipartito,ClusterNumber, pos_bip)
#HTML(ani.to_jshtml())

### Barabasi-Albert Intento de 3 clusters

In [ ]:
n_comunidad = 6
m_aristas = 2
peso_interno = 100.0
peso_corte = 1.0

G1 = nx.barabasi_albert_graph(n_comunidad, m_aristas)
G2 = nx.barabasi_albert_graph(n_comunidad, m_aristas)
G3 = nx.barabasi_albert_graph(n_comunidad, m_aristas)

mapping = {i: i+n_comunidad for i in range(n_comunidad)}
mapping2 = {i: i+n_comunidad*2 for i in range(n_comunidad)}
G2 = nx.relabel_nodes(G2, mapping)
G3 = nx.relabel_nodes(G3, mapping2)

G_bipartito = nx.union(G1,G2)
G_bipartito = nx.union(G_bipartito,G3)

for u,v in G_bipartito.edges():
  G_bipartito[u][v]['weight'] = peso_interno


# sampleo aleatorio de los dos clusters
nodosG1 = random.sample(range(len(G1)), 1)
nodosG2 = random.sample(range(len(G1), len(G1) + len(G2)), 1)
nodosG3 = random.sample(range(len(G1)*2, len(G1)*3), 1)

# creacion de uniones debiles
new_edgesG12 = list(zip(nodosG1, nodosG2))
new_edgesG13 = list(zip(nodosG1, nodosG3))
new_edgesG23 = list(zip(nodosG2, nodosG3))
G_bipartito.add_edges_from(new_edgesG12, weight=peso_corte)
G_bipartito.add_edges_from(new_edgesG13, weight=peso_corte)
G_bipartito.add_edges_from(new_edgesG23, weight=peso_corte)
A_bipartito = nx.to_numpy_array(G_bipartito)

c = 1.95
T_max = n_comunidad**2 + 1000
k = 2

ClusterNumber = FuncionOndas(c,T_max, A_bipartito,k=2)
pos_bip = nx.spring_layout(G_bipartito, k=0.65, iterations=50, seed=42)
graficador(G_bipartito, pos_bip, ClusterNumber)